<a href="https://colab.research.google.com/github/olopez27CL/Lab01_EDA_Lopez_Omar/blob/main/laboratorio_2_6/Laboratorio_2_6_Reglas_Asociacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kagglehub mlxtend

In [2]:
import os
import kagglehub
from google.colab import userdata

# Leer el token guardado en Colab Secrets
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

# Descargar el dataset de la panadería
ruta = kagglehub.dataset_download("akashdeepkuila/bakery")

print("Dataset descargado correctamente.")
print("Archivos disponibles:", os.listdir(ruta))

Using Colab cache for faster access to the 'bakery' dataset.
Dataset descargado correctamente.
Archivos disponibles: ['bakery_sales_revised.csv', 'Bakery.csv']


In [3]:
import pandas as pd

# Cargar el archivo que utilizaremos
df = pd.read_csv(os.path.join(ruta, "Bakery.csv"))

# Mostrar su tamaño y las primeras cinco filas
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])

display(df.head())

Cantidad de filas: 20507
Cantidad de columnas: 5


,TransactionNo,Items,DateTime,Daypart,DayType
0,1,Bread,2016-10-30 09:58:11,Morning,Weekend
1,2,Scandinavian,2016-10-30 10:05:34,Morning,Weekend
2,2,Scandinavian,2016-10-30 10:05:34,Morning,Weekend
3,3,Hot chocolate,2016-10-30 10:07:57,Morning,Weekend
4,3,Jam,2016-10-30 10:07:57,Morning,Weekend


In [4]:
print("Compras distintas:", df["TransactionNo"].nunique())
print("Productos distintos:", df["Items"].nunique())

Compras distintas: 9465
Productos distintos: 94


## 1. Carga y descripción del dataset

Utilicé el dataset Bakery del autor akashdeepkuila, disponible en Kaggle.
Lo descargué mediante kagglehub, utilizando un token guardado en Colab Secrets.

Los datos corresponden a compras de productos en una panadería.

- Una transacción es una compra identificada por la columna TransactionNo.
- Un ítem es un producto registrado en la columna Items.
- El archivo contiene 20.507 filas, 9.465 compras distintas y 94 productos distintos.
- Una compra puede ocupar varias filas porque puede incluir diferentes productos o repetir uno.

El dataset cumple con el requisito de tener entre 1.000 y 20.000 transacciones.
El objetivo del análisis es identificar productos que suelen comprarse juntos.

In [5]:
# Contar los valores vacíos de cada columna
print("Valores nulos por columna:")
print(df.isna().sum())

# Mostrar todos los nombres de productos
print("\nNombres de productos:")
print(sorted(df["Items"].dropna().unique()))

Valores nulos por columna:
TransactionNo    0
Items            0
DateTime         0
Daypart          0
DayType          0
dtype: int64

Nombres de productos:
['Adjustment', 'Afternoon with the baker', 'Alfajores', 'Argentina Night', 'Art Tray', 'Bacon', 'Baguette', 'Bakewell', 'Bare Popcorn', 'Basket', 'Bowl Nic Pitt', 'Bread', 'Bread Pudding', 'Brioche and salami', 'Brownie', 'Cake', 'Caramel bites', 'Cherry me Dried fruit', 'Chicken Stew', 'Chicken sand', 'Chimichurri Oil', 'Chocolates', 'Christmas common', 'Coffee', 'Coffee granules ', 'Coke', 'Cookies', 'Crepes', 'Crisps', 'Drinking chocolate spoons ', 'Duck egg', 'Dulce de Leche', 'Eggs', "Ella's Kitchen Pouches", 'Empanadas', 'Extra Salami or Feta', 'Fairy Doors', 'Farm House', 'Focaccia', 'Frittata', 'Fudge', 'Gift voucher', 'Gingerbread syrup', 'Granola', 'Hack the stack', 'Half slice Monster ', 'Hearty & Seasonal', 'Honey', 'Hot chocolate', 'Jam', 'Jammie Dodgers', 'Juice', 'Keeping It Local', 'Kids biscuit', 'Lemon and coconu

In [6]:
# Crear una copia para conservar los datos originales
df_limpio = df.copy()

# Quitar espacios sobrantes y unificar a minúsculas
df_limpio["Items"] = (
    df_limpio["Items"]
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Contar y excluir registros de ajustes administrativos
ajustes = df_limpio["Items"].eq("adjustment")
print("Filas de Adjustment excluidas:", ajustes.sum())

df_limpio = df_limpio.loc[~ajustes].copy()

print("Filas después de la limpieza:", len(df_limpio))
print("Compras distintas:", df_limpio["TransactionNo"].nunique())
print("Productos distintos:", df_limpio["Items"].nunique())

Filas de Adjustment excluidas: 1
Filas después de la limpieza: 20506
Compras distintas: 9464
Productos distintos: 93


In [7]:
# Contar repeticiones del mismo producto dentro de una compra
repetidos = df_limpio.duplicated(
    subset=["TransactionNo", "Items"]
).sum()

print("Presencias repetidas:", repetidos)

# Conservar una sola presencia por producto y compra
df_limpio = df_limpio.drop_duplicates(
    subset=["TransactionNo", "Items"]
).copy()

print("Filas después de quitar repeticiones:", len(df_limpio))
print("Compras distintas:", df_limpio["TransactionNo"].nunique())
print("Productos distintos:", df_limpio["Items"].nunique())

Presencias repetidas: 1620
Filas después de quitar repeticiones: 18886
Compras distintas: 9464
Productos distintos: 93


## 2. Revisión y limpieza de los datos

No se encontraron valores nulos en las columnas del dataset.

Se quitaron los espacios sobrantes en los nombres de productos y se
convirtieron todos los nombres a minúsculas para mantener un formato uniforme.

Se excluyó un registro llamado Adjustment, interpretado como un ajuste
administrativo. Como era el único registro de esa compra, el total de
transacciones pasó de 9.465 a 9.464.

También se quitaron 1.620 repeticiones de un mismo producto dentro de una
misma compra. Esto no significa que fueran errores: para este análisis
interesa saber si el producto está presente, sin considerar cuántas
unidades se compraron.

Después de la limpieza quedaron 18.886 filas, correspondientes a
9.464 compras distintas y 93 productos distintos.

In [8]:
# Crear una fila por compra y una columna por producto
cesta = pd.crosstab(
    df_limpio["TransactionNo"],
    df_limpio["Items"]
).gt(0)

print("Compras (filas):", cesta.shape[0])
print("Productos (columnas):", cesta.shape[1])

display(cesta.head())

Compras (filas): 9464
Productos (columnas): 93


Items,afternoon with the baker,alfajores,argentina night,art tray,bacon,baguette,bakewell,bare popcorn,basket,bowl nic pitt,...,the bart,the nomad,tiffin,toast,truffles,tshirt,valentine's card,vegan feast,vegan mincepie,victorian sponge
TransactionNo,,,,,,,,,,,,,,,,,,,,,
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 3. Codificación de las transacciones

Se creó una tabla con una fila por compra y una columna por producto.
La tabla contiene 9.464 filas y 93 columnas.

Cada celda indica si un producto está presente en una compra:
- True: el producto está presente.
- False: el producto está ausente.

Este formato se llama one-hot encoding o codificación binaria.
Apriori lo necesita para contar en cuántas compras aparece cada producto
o combinación de productos y calcular su soporte.

In [9]:
# Contar en cuántas compras aparece cada producto
frecuencias = pd.DataFrame({
    "Compras": cesta.sum(),
    "Soporte (%)": cesta.mean() * 100
}).sort_values("Compras", ascending=False)

# Mostrar los 15 productos más frecuentes
display(frecuencias.head(15).round(2))

,Compras,Soporte (%)
Items,,
coffee,4528,47.84
bread,3097,32.72
tea,1350,14.26
cake,983,10.39
pastry,815,8.61
sandwich,680,7.19
medialuna,585,6.18
hot chocolate,552,5.83
cookies,515,5.44


In [10]:
from mlxtend.frequent_patterns import apriori

# Buscar productos y combinaciones presentes en al menos el 5% de las compras
itemsets_5 = apriori(
    cesta,
    min_support=0.05,
    use_colnames=True
)

# Ordenar de mayor a menor soporte
itemsets_5 = itemsets_5.sort_values(
    "support", ascending=False
)

print("Cantidad de conjuntos frecuentes:", len(itemsets_5))
display(itemsets_5)

Cantidad de conjuntos frecuentes: 11


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
2,0.478445,(coffee)
0,0.327240,(bread)
8,0.142646,(tea)
1,0.103867,(cake)
9,0.090025,"(coffee, bread)"
6,0.086116,(pastry)
7,0.071851,(sandwich)
5,0.061813,(medialuna)
4,0.058326,(hot chocolate)
10,0.054734,"(cake, coffee)"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [11]:
from mlxtend.frequent_patterns import association_rules

reglas_5 = association_rules(
    itemsets_5,
    num_itemsets=len(cesta),
    metric="confidence",
    min_threshold=0.60
)

print("Cantidad de reglas:", len(reglas_5))

display(
    reglas_5[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ]
)

Cantidad de reglas: 0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [12]:
# Segunda configuración: soporte mínimo del 2%
itemsets_2 = apriori(
    cesta,
    min_support=0.02,
    use_colnames=True
)

# Generar reglas con confianza mínima del 40%
reglas_2 = association_rules(
    itemsets_2,
    num_itemsets=len(cesta),
    metric="confidence",
    min_threshold=0.40
)

# Ordenar por lift y luego por soporte
reglas_2 = reglas_2.sort_values(
    ["lift", "support"],
    ascending=False
)

print("Conjuntos frecuentes:", len(itemsets_2))
print("Reglas generadas:", len(reglas_2))

display(
    reglas_2[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].round(4)
)

Conjuntos frecuentes: 33
Reglas generadas: 8


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
7,(toast),(coffee),0.0237,0.7044,1.4723
4,(medialuna),(coffee),0.0352,0.5692,1.1898
5,(pastry),(coffee),0.0475,0.5521,1.1540
3,(juice),(coffee),0.0206,0.5342,1.1166
6,(sandwich),(coffee),0.0383,0.5324,1.1127
0,(cake),(coffee),0.0547,0.5270,1.1014
1,(cookies),(coffee),0.0282,0.5184,1.0836
2,(hot chocolate),(coffee),0.0296,0.5072,1.0602


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [13]:
# Tercera configuración: soporte mínimo del 1%
itemsets_1 = apriori(
    cesta,
    min_support=0.01,
    use_colnames=True
)

# Generar reglas con confianza mínima del 20%
reglas_1 = association_rules(
    itemsets_1,
    num_itemsets=len(cesta),
    metric="confidence",
    min_threshold=0.20
)

# Ordenar por lift y luego por soporte
reglas_1 = reglas_1.sort_values(
    ["lift", "support"],
    ascending=False
)

print("Conjuntos frecuentes:", len(itemsets_1))
print("Reglas generadas:", len(reglas_1))

display(
    reglas_1[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].round(4)
)

Conjuntos frecuentes: 61
Reglas generadas: 31


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
30,"(tea, coffee)",(cake),0.0100,0.2013,1.9378
12,(cake),(tea),0.0238,0.2289,1.6046
24,(toast),(coffee),0.0237,0.7044,1.4723
25,(sandwich),(tea),0.0144,0.2000,1.4021
22,(spanish brunch),(coffee),0.0109,0.5988,1.2516
16,(medialuna),(coffee),0.0352,0.5692,1.1898
18,(pastry),(coffee),0.0475,0.5521,1.1540
1,(alfajores),(coffee),0.0197,0.5407,1.1301
15,(juice),(coffee),0.0206,0.5342,1.1166
19,(sandwich),(coffee),0.0383,0.5324,1.1127


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [14]:
comparacion = pd.DataFrame({
    "Configuración": ["Exigente", "Intermedia", "Exploratoria"],
    "Soporte mínimo (%)": [5, 2, 1],
    "Confianza mínima (%)": [60, 40, 20],
    "Conjuntos frecuentes": [
        len(itemsets_5),
        len(itemsets_2),
        len(itemsets_1)
    ],
    "Reglas generadas": [
        len(reglas_5),
        len(reglas_2),
        len(reglas_1)
    ]
})

display(comparacion)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Configuración,Soporte mínimo (%),Confianza mínima (%),Conjuntos frecuentes,Reglas generadas
0,Exigente,5,60,11,0
1,Intermedia,2,40,33,8
2,Exploratoria,1,20,61,31


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 4. Justificación de los umbrales y comparación

Primero revisé la frecuencia de los productos. El café aparece en el
47,84% de las compras y el pan en el 32,72%, mientras que productos
como brownie y jugo aparecen en menos del 5%.

Comencé con un soporte mínimo del 5% para buscar combinaciones muy
frecuentes. Después lo reduje al 2% y al 1% para explorar combinaciones
menos comunes. Estos valores exigen que una combinación aparezca,
como mínimo, en 474, 190 y 95 compras, respectivamente.

Probé confianzas mínimas del 60%, 40% y 20% para comparar una selección
exigente con otras más amplias. Ordené las reglas por lift y luego
por soporte para considerar tanto la asociación como su frecuencia.

### Conclusión de la comparación

La configuración exigente produjo 11 conjuntos frecuentes y ninguna
regla. La intermedia obtuvo 33 conjuntos y 8 reglas, mientras que la
exploratoria generó 61 conjuntos y 31 reglas.

Al reducir los umbrales aparecieron más patrones. Como cambié soporte
y confianza al mismo tiempo, no puedo atribuir el cambio en la cantidad
de reglas a uno solo de estos valores.

Elegí la configuración exploratoria para seleccionar las reglas que
analizaré. Sin embargo, superar los umbrales no garantiza utilidad:
también revisaré el lift, la frecuencia y el sentido comercial.

In [15]:
# Seleccionar las cinco reglas con mayor lift
reglas_seleccionadas = reglas_1.head(5).copy()

# Preparar una tabla más fácil de leer
tabla_seleccionadas = pd.DataFrame({
    "Si la compra incluye": reglas_seleccionadas["antecedents"].apply(
        lambda productos: ", ".join(sorted(productos))
    ),
    "También incluye": reglas_seleccionadas["consequents"].apply(
        lambda productos: ", ".join(sorted(productos))
    ),
    "Soporte (%)": reglas_seleccionadas["support"] * 100,
    "Confianza (%)": reglas_seleccionadas["confidence"] * 100,
    "Lift": reglas_seleccionadas["lift"]
})

display(tabla_seleccionadas.round(2))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Si la compra incluye,También incluye,Soporte (%),Confianza (%),Lift
30,"coffee, tea",cake,1.00,20.13,1.94
12,cake,tea,2.38,22.89,1.60
24,toast,coffee,2.37,70.44,1.47
25,sandwich,tea,1.44,20.00,1.40
22,spanish brunch,coffee,1.09,59.88,1.25


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 5. Interpretación de cinco reglas

1. Café y té → pastel: el 20,13% de las compras que incluyen café y té
   también contiene pastel, por lo que podría probarse ofrecer una
   porción de pastel como complemento.

2. Pastel → té: el 22,89% de las compras con pastel también incluye té,
   lo que sugiere probar una promoción que combine ambos productos.

3. Tostada → café: el 70,44% de las compras con tostada también incluye
   café, por lo que sería una opción para probar un paquete de desayuno.

4. Sándwich → té: el 20% de las compras con sándwich también incluye té,
   por lo que podría ofrecerse esta bebida como acompañamiento.

5. Spanish brunch → café: el 59,88% de las compras con el producto
   Spanish brunch también incluye café, lo que sugiere ofrecerlos juntos.

Estas reglas describen productos que aparecen en una misma compra.
No demuestran que una promoción aumentará las ventas. Priorizaría
la prueba de tostada con café por su mayor porcentaje de compra conjunta
entre quienes compran tostada, considerando también los costos y márgenes.

In [16]:
# Buscar la regla pan → café
regla_descartada = reglas_1[
    (reglas_1["antecedents"] == frozenset({"bread"})) &
    (reglas_1["consequents"] == frozenset({"coffee"}))
]

display(
    regla_descartada[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].round(4)
)

print(
    "Porcentaje general de compras con café:",
    round(cesta["coffee"].mean() * 100, 2),
    "%"
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
4,(bread),(coffee),0.09,0.2751,0.575


Porcentaje general de compras con café: 47.84 %


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 6. Regla descartada y justificación

Descartaría la regla pan → café como base para mi primera promoción.

Aunque supera el soporte mínimo del 1% y la confianza mínima del 20%,
sus métricas muestran lo siguiente:

- Soporte: 9,00%.
- Confianza: 27,51%.
- Lift: 0,575.

El café aparece en el 47,84% de todas las compras, pero en las compras
que contienen pan esa proporción baja al 27,51%.

El lift inferior a 1 indica que ambos productos aparecen juntos
con menos frecuencia de la esperada según sus frecuencias individuales.
Por eso, que la combinación sea frecuente no significa que exista
una asociación positiva.

Preferiría probar una promoción con tostada y café, cuya regla presenta
una confianza del 70,44% y un lift de 1,47. Esto no demuestra que una
promoción de pan y café no pueda funcionar, pero los resultados
no la respaldan como primera opción.

## 7. Propuesta para el administrador

Propongo probar durante dos semanas un paquete de tostada con café
y colocar un aviso visible cerca de la caja.

Elegí esta combinación porque el 70,44% de las compras con tostada
también incluye café. Además, su lift de 1,47 muestra una asociación
positiva entre ambos productos.

Antes de definir el precio del paquete, revisaría los costos y el
margen de ganancia para evitar que un descuento reduzca la rentabilidad.

Durante la prueba registraría cuántos paquetes se venden, el valor
promedio de las compras y la ganancia obtenida. Compararía estos
resultados con días similares de semanas anteriores.

Mantendría la promoción si mejora la ganancia total. Las reglas
justifican probar la idea, pero no aseguran que aumentará las ventas.

In [17]:
# Seleccionar una muestra pequeña y reproducible
muestra = cesta.head(20)[["toast", "coffee"]].copy()

# Identificar las compras que incluyen ambos productos
muestra["ambos"] = muestra["toast"] & muestra["coffee"]

# Mostrar 1 si está presente y 0 si está ausente
display(muestra.astype(int))

print("Compras en la muestra:", len(muestra))
print("Compras con tostada:", muestra["toast"].sum())
print("Compras con café:", muestra["coffee"].sum())
print("Compras con ambos:", muestra["ambos"].sum())

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Items,toast,coffee,ambos
TransactionNo,,,
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
5,0,1,0
6,0,0,0
7,0,1,0
8,0,0,0
9,0,0,0


Compras en la muestra: 20
Compras con tostada: 0
Compras con café: 5
Compras con ambos: 0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [18]:
# Mantener las primeras 20 compras y revisar pan y café
muestra = cesta.head(20)[["bread", "coffee"]].copy()

muestra["ambos"] = muestra["bread"] & muestra["coffee"]

display(muestra.astype(int))

print("Compras en la muestra:", len(muestra))
print("Compras con pan:", muestra["bread"].sum())
print("Compras con café:", muestra["coffee"].sum())
print("Compras con ambos:", muestra["ambos"].sum())

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Items,bread,coffee,ambos
TransactionNo,,,
1,1,0,0
2,0,0,0
3,0,0,0
4,0,0,0
5,1,1,1
6,0,0,0
7,0,1,0
8,1,0,0
9,1,0,0


Compras en la muestra: 20
Compras con pan: 10
Compras con café: 5
Compras con ambos: 3


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 8. Comprobación manual sobre una muestra

Revisé la regla pan → café en las primeras 20 compras.
Conté 10 compras con pan, 5 con café y 3 con ambos productos.

- Soporte = 3 / 20 = 0,15 = 15%.
- Confianza = 3 / 10 = 0,30 = 30%.
- Lift = 0,30 / (5 / 20) = 1,20.

En esta muestra, el 30% de las compras con pan también incluye café,
mientras que el café aparece en el 25% de las compras de la muestra.

El lift de 1,20 difiere del 0,575 obtenido con el dataset completo.
Esto puede ocurrir porque las primeras 20 compras son una muestra
pequeña y no necesariamente representan todas las compras.

La comprobación permite practicar los cálculos, pero no reemplaza
los resultados obtenidos con las 9.464 transacciones.

In [19]:
# Usar solo las columnas de productos, sin la columna "ambos"
cesta_muestra = muestra[["bread", "coffee"]]

itemsets_muestra = apriori(
    cesta_muestra,
    min_support=0.05,
    use_colnames=True
)

reglas_muestra = association_rules(
    itemsets_muestra,
    num_itemsets=len(cesta_muestra),
    metric="confidence",
    min_threshold=0
)

# Mostrar únicamente la regla pan → café
verificacion = reglas_muestra[
    (reglas_muestra["antecedents"] == frozenset({"bread"})) &
    (reglas_muestra["consequents"] == frozenset({"coffee"}))
]

display(
    verificacion[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].round(4)
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
1,(bread),(coffee),0.15,0.3,1.2


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Los resultados de Apriori coinciden con los cálculos manuales realizados
sobre las mismas 20 compras.

## 9. Reflexión final

Aprendí a preparar las compras para analizar qué productos aparecen juntos.  
Al reducir los umbrales encontré más reglas, pero no todas fueron útiles.  
Comprendí que una combinación frecuente puede tener un lift inferior a 1.  
La comprobación manual coincidió con Apriori sobre la misma muestra.  
Propondría probar un paquete de tostada con café y evaluar su rentabilidad.  
Las asociaciones describen coincidencias de compra, no relaciones de causa.

In [20]:
print("Conjuntos frecuentes con soporte mínimo del 2%:")
display(
    itemsets_2.sort_values("support", ascending=False)
    .reset_index(drop=True)
)

print("Conjuntos frecuentes con soporte mínimo del 1%:")
display(
    itemsets_1.sort_values("support", ascending=False)
    .reset_index(drop=True)
)

Conjuntos frecuentes con soporte mínimo del 2%:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,support,itemsets
0,0.478445,(coffee)
1,0.327240,(bread)
2,0.142646,(tea)
3,0.103867,(cake)
4,0.090025,"(coffee, bread)"
5,0.086116,(pastry)
6,0.071851,(sandwich)
7,0.061813,(medialuna)
8,0.058326,(hot chocolate)
9,0.054734,"(cake, coffee)"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Conjuntos frecuentes con soporte mínimo del 1%:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
0,0.478445,(coffee)
1,0.327240,(bread)
2,0.142646,(tea)
3,0.103867,(cake)
4,0.090025,"(coffee, bread)"
...,...,...
56,0.010566,(hearty & seasonal)
57,0.010461,(salad)
58,0.010355,"(alfajores, bread)"
59,0.010038,"(cake, coffee, bread)"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag